# Práctica · Lectura de planes de ejecución del acueducto con Spark

**IFPN0025 · Big Data e Ingeniería de Datos** · Universidad Ean · Módulo 2 · Sesión 17

**Qué hace este cuaderno.** Lee, como evidencia, el plan de ejecución de seis consultas (los planes A, B y C y sus variantes de diagnóstico A-fecha, C-forzada y C-aqe) sobre la capa `refinada_spark/` del acueducto: cuántos archivos abre cada una, cuáles filtros se resuelven por la ruta, cuáles solo se empujan al lector y cuáles no se pueden evitar, y qué estrategia de unión usa Spark con la tabla de medidores. Cada plan se captura **normalizado y versionado**, se acompaña de sus métricas, se comprueba que las variantes dan el **mismo resultado** y que coincide con un cálculo independiente, y termina en una decisión de diseño físico justificada con su costo.

| Nivel | Nombre | Dónde | Tiempo | Qué produce |
|---|---|---|---|---|
| 1 | Guiado | En clase | 45 min | Contraste con la solución de referencia de la capa refinada; sesión `local[4]`; los planes A, B y C de la base con su interpretación; el mismo día filtrado de cuatro maneras; estadísticas de Parquet con pyarrow; unión por difusión, forzada a mezcla y con una referencia con historia; plan inicial frente a plan final; `planes_nivel1.csv` |
| 2 | Aplicado | Autónomo · obligatorio | 75 min dentro de las 3 h de la tarea | Lo mismo sobre la ampliación del equipo: cuatro funciones completadas en `pipeline/planes_s17.py`, hipótesis antes de medir, `bitacora_planes.jsonl`, `planes/` con seis planes normalizados, costo proyectado y `evidencia_nivel2_<equipo>.md` |
| 3 | Frontera | Autónomo · opcional | Fuera de las 3 h | Unión por una clave con pocos valores: predecir y medir qué tarea carga más filas y comparar tres alternativas |
| · | Reto de diseño | Autónomo · opcional | 30 min | Memo de una página: partición por día, por mes o ninguna para el cierre diario proyectado a tres años |

**Cómo usarlo.** Ejecuten las celdas en orden. El Nivel 1 está resuelto para seguirlo en clase. En el Nivel 2, el Nivel 3 y el reto encontrarán celdas con `# TODO` y `NotImplementedError` que deben completar (al terminar cada una, borren la línea `raise NotImplementedError`); después de cada ejercicio hay una celda de autoverificación con `assert` que les dice si su resultado cumple el criterio de correctitud. La lista de entrega y el `spark.stop()` finales existen solo en este cuaderno. Las decisiones de ingeniería de cada nivel están en la guía; aquí se documentan en las celdas de texto marcadas como **Su decisión**. Si reinician el núcleo, ejecuten de nuevo la configuración común, la celda de ayudantes, las de B.2 y B.3 (sesión y datos) y las definiciones de C.0.

**Entorno.** Perfil `s17` del paquete en marcha (`docker compose --profile s17 up -d`, servicio `healthy`), con este cuaderno guardado en la carpeta `practica/` y abierto desde JupyterLab (http://localhost:8888). Python 3.11, Java 11, PySpark 3.5.3, pandas y pyarrow ya vienen en la imagen. Semilla fija `SEED = 42`. La sesión usa `local[4]` y `spark.driver.memory=1g`. **Antes de abrir este cuaderno** deben estar hechos, con el cuaderno cerrado, los tres comandos de "Antes de empezar" de la guía (ampliación del equipo, su capa y su referencia): cada uno abre su propia JVM de Spark, y `spark.stop()` en un cuaderno **no libera la JVM**; con el cuaderno abierto, un script de Spark levanta una segunda JVM y el pico de memoria del contenedor sube de unos 1,7 GiB a más de 2 GiB. Los tiempos de reloj son orientativos y varían hasta ×3; lo estable son los archivos, las filas, las mezclas y los hashes.

**La regla de la práctica.** Un plan no se lee de memoria: se **predice**, se **mide** y se contrasta. Ninguna conclusión de diseño se apoya en tiempos; se apoya en archivos y bytes leídos, mezclas, y en que el resultado de las variantes sea idéntico.

### 0. Configuración común

La celda localiza el paquete, importa el módulo de planes (`pipeline/planes_s17.py`) para reutilizar sus funciones ya resueltas (sesión, captura y normalización de planes, métricas, bitácora) y calcula el hash del `datalake/` heredado: al terminar, esa carpeta debe seguir idéntica.

In [1]:
# Configuracion comun: importaciones, semilla, rutas del paquete y el modulo de planes del proyecto
import hashlib, importlib, inspect, json, os, re, subprocess, sys, time
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from pyspark.sql import functions as F

SEED = 42                                                  # semilla fija del proyecto
EQUIPO = os.environ.get("EQUIPO", "equipo-1")   # cambien por su identificador (equipo-1 a equipo-6)
RAIZ = Path.cwd().parent if Path.cwd().name == "practica" else Path.cwd()   # carpeta raiz del paquete
sys.path.insert(0, str(RAIZ / "pipeline"))
import planes_s17 as P                                     # sesion, planes, metricas y bitacora del proyecto

LAGO_BASE = RAIZ / "datalake"                              # base comun: 200 medidores, 576 000 lecturas
LAGO_EQUIPO = RAIZ / "datalake_equipo"                     # ampliacion de su equipo
HASH_LAGO_ANTES = P.hash_arbol(LAGO_BASE)                  # al terminar, el lago heredado debe seguir identico
print("Equipo:", EQUIPO, "| carpeta raiz:", RAIZ)
print("Hash del lago heredado:", HASH_LAGO_ANTES[:16], "...")

Equipo: equipo-1 | carpeta raiz: /proyecto
Hash del lago heredado: adac5febf8e7e3f5 ...


**Ayudantes.** `mostrar` imprime lo esencial de una medición, `agregar` es la agregación común (conteo y suma de consumo por sector) y `a_markdown` convierte una tabla de pandas a Markdown para la evidencia.

In [2]:
# Ayudantes del cuaderno: imprimir una medicion y correr una consulta del Nivel 1
RESULTADOS, PLANES = {}, {}      # un registro y un plan normalizado por consulta del Nivel 1 (el ultimo medido)

def mostrar(reg):
    """Lo esencial de una medicion: filtros del plan, archivos, filas, mezclas y tareas."""
    print("PartitionFilters:", reg["partition_filters"] or "(ninguno)")
    print("PushedFilters   :", reg["pushed_filters"] or "(ninguno)")
    print("ReadSchema      :", reg["read_schema"])
    print(f"archivos leidos {reg['archivos_leidos']} · carpetas {reg['carpetas_leidas']} · "
          f"tamano de los archivos {reg['tamano_archivos_leidos_b']} B ({reg['tamano_archivos_leidos_texto']})")
    print(f"filas de salida del escaneo {reg['filas_salida_escaneo']} · filas del resultado {reg['filas_resultado']}")
    print(f"union {reg['estrategia_union']} · Exchange {reg['n_exchange']} · Sort {reg['n_sort']} · "
          f"etapas {reg['etapas']} · tareas {reg['tareas']} · bytes de mezcla {reg['bytes_mezcla']}")

def agregar(df):
    """Dos medidas por sector: cuantas lecturas hay y cuanto consumo suman."""
    return df.groupBy("sector").agg(F.count("*").alias("n_lecturas"), F.sum("consumo_m3").alias("consumo_total_m3"))

def a_markdown(tabla):
    """DataFrame de pandas a tabla Markdown (sin dependencias adicionales)."""
    filas = ["| " + " | ".join(str(c) for c in tabla.columns) + " |", "|" + "---|" * len(tabla.columns)]
    filas += ["| " + " | ".join(str(v) for v in f) + " |" for f in tabla.itertuples(index=False)]
    return "\n".join(filas)

## Nivel 1 · Guiado · 45 minutos

Se resuelve sobre la **base común** (200 medidores, 120 días, 576 000 lecturas) y el día fijo **2024-02-15**. Las celdas ocupan de 1 a 2 minutos de ejecución (la primera vez que se leen los archivos desde la carpeta montada puede tardar más); el resto es lectura de planes y decisión. Si el tiempo se acaba, difieran al Nivel 2 los sub-pasos D.1b, D.3 y E.4, en ese orden.

### Sub-paso A · Contraste con la solución de referencia de la capa refinada (4 minutos)

Las dos celdas siguientes imprimen lo que el código de referencia (`pipeline/refinada_spark.py`) declara de sí mismo y lo que dejó en disco y en su bitácora. Con esa lectura y con su registro de decisiones de la práctica anterior, completen en su repositorio la tabla de contraste de la guía: una fila por decisión, con lo que mantienen, lo que cambiarían y por qué, y con la pregunta nueva: **¿qué consulta de las tres de hoy se vuelve más cara con la decisión de escritura que tomé?** Una decisión distinta de la de referencia no es incorrecta si está bien justificada. Si su registro todavía no está terminado, contrasten las cuatro decisiones con lo que su código de la capa hace hoy, escriban decisiones provisionales con la mejor cifra disponible y declárenlo; lo que no alcance pasa al Nivel 2.

In [3]:
# Sub-paso A (1/2): lo que la solucion de referencia de la capa refinada declara de si misma.
# Se imprimen las lineas que fijan cada decision de escritura: una decision de escritura es una decision de lectura futura.
import refinada_spark as R
for nombre in ("columnas_derivadas", "escribir_capa"):
    print(f"--- {nombre} ---")
    for linea in inspect.getsource(getattr(R, nombre)).splitlines():
        if not linea.strip().startswith("-") and any(clave in linea for clave in ("withColumn", "partitionBy", "outputTimestampType", "partitionOverwriteMode", "repartition")):
            print(linea.rstrip())

--- columnas_derivadas ---
    TODO (equipo): agreguen estas columnas con withColumn, todas a partir de las existentes:
    return (df.withColumn("fecha", F.to_date(ts))
              .withColumn("hora", F.hour(ts))
              .withColumn("es_confiable", F.col("quality_flag") == F.lit("OK"))
              .withColumn("anio", F.date_format(ts, "yyyy"))
              .withColumn("mes", F.date_format(ts, "MM"))
              .withColumn("dia", F.date_format(ts, "dd")))
--- escribir_capa ---
    Decision (equipo-1): un archivo por dia (repartition por anio, mes y dia antes de escribir):
    spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
    spark.conf.set("spark.sql.sources.partitionOverwriteMode",
    (df.repartition("anio", "mes", "dia")
       .partitionBy("anio", "mes", "dia")


In [4]:
# Sub-paso A (2/2): lo que dejo escrito en disco y en su bitacora la ejecucion de referencia sobre la base comun
capa_ref = LAGO_BASE / "refinada_spark"
archivos = sorted(capa_ref.rglob("*.parquet"))
tam = [a.stat().st_size for a in archivos]
print(f"{len(archivos)} archivos .parquet, uno por dia · {sum(tam) / 1024:.1f} KiB en total · {sum(tam) / len(tam) / 1024:.2f} KiB por archivo")
columnas_archivo = pq.ParquetFile(archivos[0]).schema.names
print("columnas guardadas DENTRO del archivo:", columnas_archivo)
print("columnas que solo estan en la RUTA   :", [c for c in ("anio", "mes", "dia") if c not in columnas_archivo])
print("ruta de ejemplo:", archivos[0].relative_to(capa_ref).as_posix())
corridas = [json.loads(l) for l in (RAIZ / "bitacora_refinada_spark.jsonl").read_text(encoding="utf-8").splitlines() if '"hash_salida"' in l]
ultima = [c for c in corridas if c["volumen"] == "base"][-1]          # la corrida de la solucion de referencia sobre la base comun
print("bitacora de la capa:", {k: ultima[k] for k in ("volumen", "filas_escritas", "archivos_escritos", "sobrescritura", "reconciliacion_heredada", "reconciliacion_control")})

120 archivos .parquet, uno por dia · 2935.8 KiB en total · 24.47 KiB por archivo
columnas guardadas DENTRO del archivo: ['meter_id', 'timestamp', 'sector', 'consumo_m3', 'presion_bar', 'quality_flag', 'fecha', 'hora', 'es_confiable', 'nivel_presion']
columnas que solo estan en la RUTA   : ['anio', 'mes', 'dia']
ruta de ejemplo: anio=2024/mes=01/dia=01/part-00000-ad105644-13dd-4a6f-ba3b-b136b44d23c0.c000.snappy.parquet
bitacora de la capa: {'volumen': 'base', 'filas_escritas': 576000, 'archivos_escritos': 120, 'sobrescritura': 'dinamica', 'reconciliacion_heredada': 'OK', 'reconciliacion_control': 'OK'}


**Su decisión · Sub-paso A (equipo-1).**

| Decisión de escritura de la capa (T16) | ¿Qué mantenemos o cambiamos hoy? | Consulta de hoy que se encarece |
|---|---|---|
| `anio`, `mes` y `dia` como texto con ceros, derivados de la marca temporal | **Mantenemos.** Con texto, `PartitionFilters` poda: A lee 1 archivo. Con enteros inferidos, un literal `'02'` y una columna `2` no coinciden (D.1b), y en T16 vimos duplicados con `mes=1` junto a `mes=01`. | Ninguna, si el literal tiene el tipo correcto. |
| Un archivo por día | **Mantenemos.** Nuestra T16 sí usa `repartition(anio, mes, dia)`, a diferencia de la referencia, para garantizar exactamente un archivo por carpeta. | El **reporte mensual**: abre 31 archivos pequeños (1 140 KiB) en lugar de 1. |
| `fecha` y `hora` dentro del archivo, y no solo en la ruta | **Mantenemos.** Son útiles para leer, pero filtrar por `fecha` **no poda**: A-fecha abre 120 archivos. | **A-fecha**: 120 archivos en lugar de 1, por el mismo resultado. |
| La capa conserva todas las filas, con `es_confiable` | **Mantenemos.** Es la condición para reconciliar sin filtro. | **C**: el filtro `es_confiable = true` solo se empuja, y se leen las 1 002 240 filas (los 120 archivos). |


### Sub-paso B · Perfil `s17` y sesión de Spark (3 minutos)

Desde la terminal, **antes** de abrir el cuaderno: `docker compose --profile s12 down` y luego `docker compose --profile s17 up -d`. `s17` es **el mismo servicio** `spark` de las dos prácticas anteriores con una etiqueta de perfil más: el contenedor sigue llamándose `acueducto_spark_s15` y la imagen `acueducto-spark:s15`; los comandos se dirigen al servicio (`exec spark`), no al nombre del contenedor. En la sesión `local[4]` todo vive en una sola JVM: el único "ejecutor" es el propio controlador. Mientras la sesión esté viva, la interfaz de Spark se abre en http://localhost:4040: la pestaña **SQL / DataFrame** muestra el plan **ejecutado** de cada consulta con sus métricas (archivos leídos, filas de salida, bytes de mezcla), y desaparece al detener la sesión. La celda B.2 muestra primero los valores **por omisión** de Spark (ejecución adaptativa activa, 200 particiones de mezcla, inferencia de tipo de las columnas de partición activa), que son los que verán en cualquier otro entorno, y luego fija los de la práctica.

In [5]:
# B.1 Comprobacion del entorno: versiones que corren dentro del contenedor
import pyspark, pyarrow
java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0].split(" 20")[0]   # solo nombre y version
print("Python", sys.version.split()[0], "· PySpark", pyspark.__version__, "· pandas", pd.__version__, "· pyarrow", pyarrow.__version__)
print(java)

Python 3.11.17 · PySpark 3.5.3 · pandas 2.2.2 · pyarrow 15.0.2
openjdk version "11.0.32.1"


In [6]:
# B.2 La sesion de Spark. local[4] = una sola JVM con 4 ranuras de tareas; crear_sesion fija UTC y la memoria.
# Primero se leen los valores POR OMISION de Spark (los que veran en cualquier otro entorno) y luego se fija
# lo que cambia el plan: ejecucion adaptativa apagada, 8 particiones de mezcla y columnas de particion como texto.
spark = P.crear_sesion("local[4]", "1g")
CLAVES = ["spark.sql.adaptive.enabled", "spark.sql.shuffle.partitions", "spark.sql.autoBroadcastJoinThreshold",
          "spark.sql.sources.partitionColumnTypeInference.enabled", "spark.sql.cbo.enabled"]
print("Por omision en Spark", spark.version)
for k in CLAVES:
    print(f"  {k} = {spark.conf.get(k)}")
P.configurar(spark)
print("En la practica")
for k in CLAVES:
    print(f"  {k} = {spark.conf.get(k)}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 22:17:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Por omision en Spark 3.5.3


  spark.sql.adaptive.enabled = true
  spark.sql.shuffle.partitions = 200
  spark.sql.autoBroadcastJoinThreshold = 10485760b
  spark.sql.sources.partitionColumnTypeInference.enabled = true
  spark.sql.cbo.enabled = false
En la practica
  spark.sql.adaptive.enabled = false
  spark.sql.shuffle.partitions = 8
  spark.sql.autoBroadcastJoinThreshold = 10485760b
  spark.sql.sources.partitionColumnTypeInference.enabled = false
  spark.sql.cbo.enabled = false


In [7]:
# B.3 Leer la capa y la referencia UNA vez por sesion: listar 120 carpetas y leer el esquema cuesta segundos.
DIA = "2024-02-15"                      # dia fijo del Nivel 1 (dia 46 de 120)
t0 = time.perf_counter()
datos = P.preparar_datos(spark, LAGO_BASE, dia=DIA)
capa = datos["capa_df"]
print(f"capa y referencia leidas en {time.perf_counter() - t0:.1f} s")
print("tipos de las columnas de particion:", [t for t in capa.dtypes if t[0] in ("anio", "mes", "dia")])
print("referencia de medidores:", datos["referencia_df"].count(), "filas")

capa y referencia leidas en 1.1 s
tipos de las columnas de particion: [('anio', 'string'), ('mes', 'string'), ('dia', 'string')]


referencia de medidores: 200 filas


In [8]:
# B.4 Pestana Executors (por su API REST): en modo local hay UN solo ejecutor, llamado "driver"
for e in P._rest(spark, "executors"):
    print(f"ejecutor {e['id']} · ranuras {e['totalCores']} · memoria de almacenamiento {e['maxMemory'] / 2**20:.1f} MiB")

ejecutor driver · ranuras 4 · memoria de almacenamiento 434.4 MiB


In [9]:
# Autoverificacion del sub-paso B
assert spark.conf.get("spark.sql.adaptive.enabled") == "false"
assert spark.conf.get("spark.sql.shuffle.partitions") == "8"
assert P.umbral_actual(spark) == 10 * 1024 * 1024
assert dict(capa.dtypes)["mes"] == "string"            # anio, mes y dia llegan como texto con ceros
assert capa.count() == 576_000
print("Sub-paso B: OK")

Sub-paso B: OK


**Su decisión · Sub-paso B (decisión 5).** Fijamos `spark.sql.adaptive.enabled=false`, `spark.sql.shuffle.partitions=8` y la inferencia de tipos de las columnas de partición desactivada. Con los valores por omisión:
- **Ejecución adaptativa activa:** el plan que muestra `explain` es solo el inicial, porque el final cambia al ejecutar (en C-aqe, el hash inicial `fe97…` es distinto del final `f0f5…` y las tareas bajan de 13 a 6).
- **200 particiones de mezcla:** 200 tareas casi vacías para 4 sectores.
- **Inferencia de tipos de partición:** `mes` se leería como el entero `2`, y un literal `'02'` dejaría de podar o daría un resultado vacío.

Con cualquiera de los tres, ni el hash del plan ni las tareas serían comparables entre corridas ni entre equipos.

### Sub-paso C · Los planes A y B (10 minutos)

**C.0** define las consultas del nivel (cada una recibe la capa, la referencia y el día). **C.1** ejecuta el plan A del día 2024-02-15 (y **C.1b** compara su plan lógico analizado con el optimizado): el árbol del plan físico se lee **de abajo hacia arriba** (las hojas son lecturas; la raíz, el resultado) y `Exchange` es una mezcla y, por tanto, un límite de etapa. **C.2** ejecuta el plan B: un predicado sobre una columna sin partición ni orden. **C.3** lee las lecturas confiables sin unir, para ver `ReadSchema`: solo las columnas que la consulta necesita.

In [10]:
# C.0 Las consultas del Nivel 1. Cada una recibe (capa, medidores, dia) y devuelve un DataFrame.
def q_a(capa, medidores, dia):
    """A: consumo por sector de UN dia, filtrando por las columnas de la ruta (anio, mes, dia)."""
    anio, mes, d = dia.split("-")
    return agregar(capa.filter((F.col("anio") == anio) & (F.col("mes") == mes) & (F.col("dia") == d)))

def q_a_fecha(capa, medidores, dia):
    """A-fecha: el mismo dia, filtrado por la columna fecha, que esta guardada DENTRO del archivo."""
    return agregar(capa.filter(F.col("fecha") == F.lit(dia).cast("date")))

def q_b(capa, medidores, dia):
    """B: lecturas con presion alta (mas de 3,5 bar) por sector."""
    return agregar(capa.filter(F.col("presion_bar") > 3.5))

def q_c(capa, medidores, dia):
    """C: consumo por estrato de las lecturas confiables, uniendo con la referencia de medidores."""
    lecturas = capa.filter(F.col("es_confiable")).select("meter_id", "consumo_m3")
    return (lecturas.join(medidores.select("meter_id", "estrato"), "meter_id")
                    .groupBy("estrato").agg(F.count("*").alias("n_lecturas"), F.sum("consumo_m3").alias("consumo_total_m3")))

Q1 = {"A": q_a, "A-fecha": q_a_fecha, "B": q_b, "C": q_c, "C-forzada": q_c, "C-aqe": q_c}

def correr(nombre):
    """Ejecuta una consulta del Nivel 1 con la configuracion que le corresponde (C-forzada: umbral -1;
    C-aqe: ejecucion adaptativa activa) y guarda su registro en RESULTADOS."""
    reg, plan, pdf = P.ejecutar_consulta(spark, nombre, datos, {}, 1, EQUIPO, "base", constructores=Q1)
    RESULTADOS[nombre], PLANES[nombre] = reg, plan
    return reg, plan, pdf
print("Consultas definidas:", list(Q1))

Consultas definidas: ['A', 'A-fecha', 'B', 'C', 'C-forzada', 'C-aqe']


In [11]:
# C.1 Plan A del dia 2024-02-15: el arbol del plan fisico (de abajo hacia arriba) y los campos de la lectura
reg, plan, resultado = correr("A")
print(P.arbol_del_plan(plan).strip())
print()
mostrar(reg)
print(resultado.sort_values("sector").round(3).to_string(index=False))

== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Project (3)
         +- * ColumnarToRow (2)
            +- Scan parquet  (1)

PartitionFilters: [isnotnull(anio), isnotnull(mes), isnotnull(dia), (anio = 2024), (mes = 02), (dia = 15)]
PushedFilters   : (ninguno)
ReadSchema      : struct<sector:string,consumo_m3:double>
archivos leidos 1 · carpetas 1 · tamano de los archivos 24923 B (24.3 KiB)
filas de salida del escaneo 4800 · filas del resultado 4
union None · Exchange 1 · Sort 0 · etapas 2 · tareas 9 · bytes de mezcla 209
 sector  n_lecturas  consumo_total_m3
 S-ESTE        1200           192.831
S-NORTE        1200           192.464
S-OESTE        1200           193.830
  S-SUR        1200           192.724


In [12]:
# C.1b El plan LOGICO de A, antes y despues de optimizar (explain extended, recortado): el optimizador reescribe el arbol
# sin cambiar el resultado. Compare el analizado (lo que se pidio) con el optimizado (lo que se ejecutara).
texto = P.capturar_plan(q_a(capa, datos["referencia_df"], DIA), "extended")
limpiar = lambda s: "\n".join(l[:150] for l in re.sub(r"#\d+L?", "", s).strip().splitlines())
analizado = limpiar(texto.split("== Analyzed Logical Plan ==")[1].split("== Optimized Logical Plan ==")[0])
optimizado = limpiar(texto.split("== Optimized Logical Plan ==")[1].split("== Physical Plan ==")[0])
print("--- Plan logico analizado ---"); print(analizado)
print("--- Plan logico optimizado ---"); print(optimizado)
assert "Project" in optimizado and "Project" not in analizado       # la poda de columnas agrega un Project
assert "isnotnull(anio)" in optimizado                               # el optimizador infiere que la ruta no es nula

--- Plan logico analizado ---
sector: string, n_lecturas: bigint, consumo_total_m3: double
Aggregate [sector], [sector, count(1) AS n_lecturas, sum(consumo_m3) AS consumo_total_m3]
+- Filter (((anio = 2024) AND (mes = 02)) AND (dia = 15))
   +- Relation [meter_id,timestamp,sector,consumo_m3,presion_bar,quality_flag,fecha,hora,es_confiable,nivel_presion,anio,mes,dia] parquet
--- Plan logico optimizado ---
Aggregate [sector], [sector, count(1) AS n_lecturas, sum(consumo_m3) AS consumo_total_m3]
+- Project [sector, consumo_m3]
   +- Filter (((isnotnull(anio) AND isnotnull(mes)) AND isnotnull(dia)) AND (((anio = 2024) AND (mes = 02)) AND (dia = 15)))
      +- Relation [meter_id,timestamp,sector,consumo_m3,presion_bar,quality_flag,fecha,hora,es_confiable,nivel_presion,anio,mes,dia] parquet


In [13]:
# C.2 Plan B: presion alta. El predicado aparece empujado al lector; que ahorre archivos es otra pregunta
reg, plan, resultado = correr("B")
print(P.arbol_del_plan(plan).strip())
print()
mostrar(reg)
print("lecturas con presion alta:", int(resultado["n_lecturas"].sum()), "·", resultado.set_index("sector")["n_lecturas"].to_dict())

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)

PartitionFilters: (ninguno)
PushedFilters   : [IsNotNull(presion_bar), GreaterThan(presion_bar,3.5)]
ReadSchema      : struct<sector:string,consumo_m3:double,presion_bar:double>
archivos leidos 120 · carpetas 120 · tamano de los archivos 3006308 B (2935.8 KiB)
filas de salida del escaneo 576000 · filas del resultado 4
union None · Exchange 1 · Sort 0 · etapas 2 · tareas 12 · bytes de mezcla 864
lecturas con presion alta: 27472 · {'S-ESTE': 6805, 'S-SUR': 6902, 'S-OESTE': 6819, 'S-NORTE': 6946}


In [14]:
# C.3 Lecturas confiables, todavia sin union. El select pide meter_id, pero la agregacion no lo usa: Catalyst lo poda
# y ReadSchema trae solo dos columnas (poda de columnas)
df = capa.filter(F.col("es_confiable")).select("meter_id", "consumo_m3").agg(F.count("*").alias("n_lecturas"), F.sum("consumo_m3").alias("consumo_total_m3"))
reg, plan, _, resultado = P.medir_df(spark, df)
mostrar(reg)
print("lecturas confiables:", int(resultado["n_lecturas"].iloc[0]))

PartitionFilters: (ninguno)
PushedFilters   : [IsNotNull(es_confiable), EqualTo(es_confiable,true)]
ReadSchema      : struct<consumo_m3:double,es_confiable:boolean>
archivos leidos 120 · carpetas 120 · tamano de los archivos 3006308 B (2935.8 KiB)
filas de salida del escaneo 576000 · filas del resultado 1
union None · Exchange 1 · Sort 0 · etapas 2 · tareas 5 · bytes de mezcla 260
lecturas confiables: 541368


In [15]:
# Autoverificacion del sub-paso C: lo que se debe cumplir; las cifras medidas se informan
a, b = RESULTADOS["A"], RESULTADOS["B"]
assert a["archivos_leidos"] == 1 and all(c in a["partition_filters"] for c in ("anio", "mes", "dia"))
assert b["partition_filters"] == "" and "presion_bar" in b["pushed_filters"]
assert b["archivos_leidos"] == 120 and b["filas_salida_escaneo"] == 576_000    # empujar no es evitar
assert reg["read_schema"].count(":") == 2                                     # dos columnas leidas: meter_id se podo
print("Sub-paso C: OK · A lee", a["archivos_leidos"], "archivo y B lee", b["archivos_leidos"])

Sub-paso C: OK · A lee 1 archivo y B lee 120


**Su decisión · Sub-paso C (decisión 2).**
- **Qué resolvió cada filtro:** en **A**, el filtro se resolvió por la **ruta**: `PartitionFilters [(anio = 2024), (mes = 02), (dia = 15)]`, que evitó 119 de 120 archivos. En **B**, el predicado `presion_bar > 3,5` se **empujó al lector**: `PushedFilters [GreaterThan(presion_bar,3.5)]`, que no evitó **ningún** archivo (120 de 120).
- **Por qué empujar no reduce la lectura:** `PushedFilters` solo dice que el lector **puede** usar el predicado para saltar grupos de filas comparándolo con sus estadísticas. Como cada archivo tiene un solo grupo de filas y su máximo de presión es de al menos 3,951 bar en la base, ninguna estadística permite saltarlo: se leen las 576 000 filas.

### Sub-paso D · Por qué la poda funciona o falla (8 minutos)

**D.1** filtra el mismo día de cuatro maneras y cuenta los archivos que se abren: la poda depende de **de qué columnas depende el filtro**, no de su forma. **D.1b** muestra el fallo silencioso del literal: con columnas de texto, `'2'` no coincide con `'02'` y el resultado es vacío sin error. **D.2** lee con pyarrow, sin Spark, las estadísticas de cada archivo y los bytes por columna: explican por qué se salta (o no) un grupo de filas. **D.3** muestra qué se pierde al leer la carpeta de un solo día sin `basePath`.

In [16]:
# D.1 El MISMO dia (4 800 lecturas) filtrado de cuatro maneras: cuantos archivos se abren en cada una?
a, m, d = DIA.split("-")
reg_fecha, _, _ = correr("A-fecha")
medidas = {"ruta (anio, mes, dia)": (RESULTADOS["A"], None), "fecha (columna dentro del archivo)": (reg_fecha, None)}
casos = {"concat de las columnas de la ruta": F.concat(F.col("anio"), F.col("mes"), F.col("dia")) == a + m + d,
         "ruta del archivo (input_file_name)": F.input_file_name().contains(f"anio={a}/mes={m}/dia={d}")}
for etiqueta, condicion in casos.items():
    r, _, _, pdf = P.medir_df(spark, agregar(capa.filter(condicion)))
    medidas[etiqueta] = (r, pdf)
tabla = pd.DataFrame([{"forma de filtrar": k, "poda por la ruta": "si" if r["partition_filters"] else "no", "archivos": r["archivos_leidos"],
                       "filas de salida del escaneo": r["filas_salida_escaneo"], "filas del resultado": r["filas_resultado"],
                       "hash del resultado": r["hash_resultado"][:8]} for k, (r, _) in medidas.items()])
print(tabla.to_string(index=False))
print("PartitionFilters de concat:", medidas["concat de las columnas de la ruta"][0]["partition_filters"])
print("tamano de los archivos leidos con fecha:", reg_fecha["tamano_archivos_leidos_b"], "B (todos los archivos, aunque salto 119 grupos de filas)")

                  forma de filtrar poda por la ruta  archivos  filas de salida del escaneo  filas del resultado hash del resultado
             ruta (anio, mes, dia)               si         1                         4800                    4           a21da324
fecha (columna dentro del archivo)               no       120                         4800                    4           a21da324
 concat de las columnas de la ruta               si         1                         4800                    4           a21da324
ruta del archivo (input_file_name)               no       120                       576000                    4           a21da324
PartitionFilters de concat: [(concat(anio, mes, dia) = 20240215)]
tamano de los archivos leidos con fecha: 3006308 B (todos los archivos, aunque salto 119 grupos de filas)


In [17]:
# D.1b El literal de un filtro sobre la ruta: tipo y ceros. Con columnas de texto, '2' no coincide con '02'
def filtrar_dia(capa_x, mes_literal):
    return agregar(capa_x.filter((F.col("anio") == "2024") & (F.col("mes") == mes_literal) & (F.col("dia") == "15")))

literales = []
for etiqueta, literal in {"mes = '02' (texto con cero)": "02", "mes = 2 (entero)": 2, "mes = '2' (texto sin cero)": "2"}.items():
    r, _, _, pdf = P.medir_df(spark, filtrar_dia(capa, literal))
    literales.append({"filtro": etiqueta, "archivos": r["archivos_leidos"], "lecturas devueltas": int(pdf["n_lecturas"].sum()) if len(pdf) else 0})
print(pd.DataFrame(literales).to_string(index=False))
# Con la inferencia de tipo de Spark activada (su valor por omision) mes se lee como entero y '2' si coincide
P.configurar(spark, inferir_tipos=True)
capa_inferida = P.leer_capa(spark, datos["capa"])
r, _, _, pdf = P.medir_df(spark, filtrar_dia(capa_inferida, "2"))
print("con inferencia activada: mes es", dict(capa_inferida.dtypes)["mes"], "· mes = '2' devuelve", int(pdf["n_lecturas"].sum()), "lecturas")
P.configurar(spark)

                     filtro  archivos  lecturas devueltas
mes = '02' (texto con cero)         1                4800
           mes = 2 (entero)         1                4800
 mes = '2' (texto sin cero)         0                   0


con inferencia activada: mes es int · mes = '2' devuelve 4800 lecturas


In [18]:
# D.2 Lector independiente: estadisticas por archivo y bytes por columna, leidas con pyarrow (sin Spark)
estadisticas, por_columna = P.estadisticas_parquet(datos["capa"])
print(estadisticas.head(3).to_string(index=False))
print("grupos de filas por archivo:", sorted(estadisticas["grupos_de_filas"].unique()), "· filas por grupo:", sorted(estadisticas["filas"].unique()))
print("fecha: minimo igual a maximo en", int((estadisticas["fecha_min"] == estadisticas["fecha_max"]).sum()), "de", len(estadisticas), "archivos")
print("presion_bar: el menor de los maximos", round(estadisticas["presion_bar_max"].min(), 3), "· el mayor", round(estadisticas["presion_bar_max"].max(), 3))
for umbral in (3.5, 4.0, 4.2):
    print(f"  archivos que un umbral de {umbral} bar permitiria saltar: {int((estadisticas['presion_bar_max'] <= umbral).sum())} de {len(estadisticas)}")
print(por_columna.head(5).to_string(index=False))
frac = por_columna.set_index("columna")["fraccion"]
print(f"fraccion de bytes de A (sector, consumo_m3): {frac[['sector', 'consumo_m3']].sum():.1%} · de B (+ presion_bar): {frac[['sector', 'consumo_m3', 'presion_bar']].sum():.1%}")

archivo  grupos_de_filas  filas  presion_bar_min  presion_bar_max  fecha_min  fecha_max
 dia=01                1   4800            1.977            4.208 2024-01-01 2024-01-01
 dia=02                1   4800            1.925            4.005 2024-01-02 2024-01-02
 dia=03                1   4800            1.934            4.298 2024-01-03 2024-01-03
grupos de filas por archivo: [1] · filas por grupo: [4800]
fecha: minimo igual a maximo en 120 de 120 archivos
presion_bar: el menor de los maximos 3.951 · el mayor 4.404
  archivos que un umbral de 3.5 bar permitiria saltar: 0 de 120
  archivos que un umbral de 4.0 bar permitiria saltar: 11 de 120
  archivos que un umbral de 4.2 bar permitiria saltar: 101 de 120
      columna  bytes_comprimidos  fraccion
  presion_bar            1420799    0.5258
   consumo_m3             745056    0.2757
     meter_id             153600    0.0568
nivel_presion             121004    0.0448
 quality_flag              93246    0.0345
fraccion de bytes de A (

In [19]:
# D.3 basePath: leer la carpeta de UN solo dia pierde las columnas de la ruta si no se declara
carpeta_dia = datos["capa"] / f"anio={a}" / f"mes={m}" / f"dia={d}"
sin_base = spark.read.parquet(str(carpeta_dia))
con_base = spark.read.option("basePath", str(datos["capa"])).parquet(str(carpeta_dia))
print("sin basePath:", len(sin_base.columns), "columnas · anio, mes y dia presentes:", all(c in sin_base.columns for c in ("anio", "mes", "dia")))
print("con basePath:", len(con_base.columns), "columnas · anio, mes y dia presentes:", all(c in con_base.columns for c in ("anio", "mes", "dia")))
print("filas de ese dia:", con_base.count())
# Particiones de lectura: Spark agrupa los archivos seleccionados; cada archivo ademas tiene un costo de apertura
print("particiones de lectura · capa completa:", capa.rdd.getNumPartitions(), "· un mes (31 archivos):", capa.filter(F.col("mes") == "01").rdd.getNumPartitions(),
      "· un solo dia:", capa.filter((F.col("mes") == "02") & (F.col("dia") == "15")).rdd.getNumPartitions())

sin basePath: 10 columnas · anio, mes y dia presentes: False
con basePath: 13 columnas · anio, mes y dia presentes: True


filas de ese dia: 4800


particiones de lectura · capa completa: 4 · un mes (31 archivos): 4 · un solo dia: 1


In [20]:
# Autoverificacion del sub-paso D
assert medidas["ruta (anio, mes, dia)"][0]["archivos_leidos"] == 1
assert medidas["fecha (columna dentro del archivo)"][0]["archivos_leidos"] == 120
assert medidas["fecha (columna dentro del archivo)"][0]["filas_salida_escaneo"] == 4_800     # el lector salto 119 grupos
assert medidas["concat de las columnas de la ruta"][0]["archivos_leidos"] == 1               # una funcion de la ruta tambien poda
assert medidas["ruta del archivo (input_file_name)"][0]["archivos_leidos"] == 120
assert len({r["hash_resultado"] for r, _ in medidas.values()}) == 1                          # el mismo resultado en las cuatro
assert len(sin_base.columns) + 3 == len(con_base.columns)
print("Sub-paso D: OK")

Sub-paso D: OK


**Su decisión · Sub-paso D (decisión 2).** Para el cierre diario recomendamos filtrar por **`anio`, `mes` y `dia` como texto con ceros**, con literales del mismo tipo. De las cuatro formas de filtrar el mismo día:
- **Podan:** la **ruta** (1 archivo) y **`concat(anio, mes, dia)`**, porque Catalyst la deja en `PartitionFilters` (1 archivo).
- **No podan:** **`fecha`** (120 archivos abiertos, aunque salta 119 grupos de filas) e **`input_file_name()`**, que abre los 120 archivos y escanea las 576 000 filas.
- **Literal sin ceros:** filtrar `mes = '2'` en lugar de `'02'` devuelve vacío sin error. Por eso el tipo y los ceros del literal son parte de la decisión.

### Sub-paso E · Estrategias de unión (13 minutos)

**E.1** ejecuta C por omisión: Catalyst decide con el **tamaño estimado** de cada lado y difunde el menor. **E.2** desactiva el umbral para ver la mezcla con ordenamiento y su costo. **E.3** une con una referencia con historia: una clave que no es única multiplica filas. **E.4** activa la ejecución adaptativa y compara el plan inicial con el final (`C-aqe`): el plan puede cambiar al ejecutar, y por eso la evidencia es el plan más las métricas más un resultado idéntico, no el tiempo.

In [21]:
# E.1 Consulta C por omision: Catalyst elige la estrategia de union segun el TAMANO ESTIMADO de cada lado
reg, plan, resultado = correr("C")
print(P.arbol_del_plan(plan).strip())
print()
mostrar(reg)
izq, der = P.tamanos_estimados(Q1["C"](capa, datos["referencia_df"], DIA))
print(f"tamano estimado: lado de lecturas {izq} B · lado de la referencia {der} B · umbral de difusion {P.umbral_actual(spark)} B")
print(f"tamano real de lo difundido (data size del BroadcastExchange): {reg['difusion_bytes']} B · la referencia pesa "
      f"{(LAGO_BASE / 'referencia' / 'medidores.parquet').stat().st_size} B en disco")
print(resultado.sort_values("estrato").round(3).to_string(index=False))

== Physical Plan ==
* HashAggregate (13)
+- Exchange (12)
   +- * HashAggregate (11)
      +- * Project (10)
         +- * BroadcastHashJoin Inner BuildRight (9)
            :- * Project (4)
            :  +- * Filter (3)
            :     +- * ColumnarToRow (2)
            :        +- Scan parquet  (1)
            +- BroadcastExchange (8)
               +- * Filter (7)
                  +- * ColumnarToRow (6)
                     +- Scan parquet  (5)

PartitionFilters: (ninguno)
PushedFilters   : [IsNotNull(es_confiable), EqualTo(es_confiable,true), IsNotNull(meter_id)]
ReadSchema      : struct<meter_id:string,consumo_m3:double,es_confiable:boolean>
archivos leidos 120 · carpetas 120 · tamano de los archivos 3006308 B (2935.8 KiB)
filas de salida del escaneo 576000 · filas del resultado 4
union BroadcastHashJoin · Exchange 1 · Sort 0 · etapas 3 · tareas 13 · bytes de mezcla 707
tamano estimado: lado de lecturas 597939 B · lado de la referencia 1639 B · umbral de difusion 10485760 B
ta

In [22]:
# E.2 La misma consulta con el umbral de difusion desactivado (-1): ahora hay que repartir y ordenar ambos lados
reg_f, plan_f, resultado_f = correr("C-forzada")
print(P.arbol_del_plan(plan_f).strip())
print()
comparacion = pd.DataFrame([{"consulta": n, "union": r["estrategia_union"], "Exchange": r["n_exchange"], "Sort": r["n_sort"], "etapas": r["etapas"],
                             "tareas": r["tareas"], "bytes de mezcla": r["bytes_mezcla"], "hash del resultado": r["hash_resultado"][:8]}
                            for n, r in (("C", RESULTADOS["C"]), ("C-forzada", reg_f))])
print(comparacion.to_string(index=False))

== Physical Plan ==
* HashAggregate (16)
+- Exchange (15)
   +- * HashAggregate (14)
      +- * Project (13)
         +- * SortMergeJoin Inner (12)
            :- * Sort (6)
            :  +- Exchange (5)
            :     +- * Project (4)
            :        +- * Filter (3)
            :           +- * ColumnarToRow (2)
            :              +- Scan parquet  (1)
            +- * Sort (11)
               +- Exchange (10)
                  +- * Filter (9)
                     +- * ColumnarToRow (8)
                        +- Scan parquet  (7)

 consulta             union  Exchange  Sort  etapas  tareas  bytes de mezcla hash del resultado
        C BroadcastHashJoin         1     0       3      13              707           cf38f78f
C-forzada     SortMergeJoin         3     2       4      21          4391404           cf38f78f


In [23]:
# E.3 Una referencia con historia (un medidor aparece dos veces) multiplica filas: la clave no es unica
historia = P.leer_referencia(spark, LAGO_BASE, con_historia=True)
todas = capa.select("meter_id", "consumo_m3")

def unir(referencia):
    return todas.join(referencia.select("meter_id", "estrato"), "meter_id").agg(F.count("*").alias("filas"), F.sum("consumo_m3").alias("consumo")).toPandas().iloc[0]

en_capa = todas.agg(F.count("*").alias("filas"), F.sum("consumo_m3").alias("consumo")).toPandas().iloc[0]
vigente, con_historia = unir(datos["referencia_df"]), unir(historia)
print(pd.DataFrame({"capa": en_capa, "union con la referencia vigente": vigente, "union con la referencia con historia": con_historia}).round(3).to_string())
print("filas de mas:", int(con_historia["filas"] - en_capa["filas"]), "· consumo de mas:", round(float(con_historia["consumo"] - en_capa["consumo"]), 3), "m3")
repetidos = historia.groupBy("meter_id").count().filter("count > 1").toPandas()
print("medidores con mas de una fila en la referencia:", repetidos.to_dict("records"))

               capa  union con la referencia vigente  union con la referencia con historia
filas    576000.000                       576000.000                            578880.000
consumo   92166.022                        92166.022                             92627.633
filas de mas: 2880 · consumo de mas: 461.611 m3


medidores con mas de una fila en la referencia: [{'meter_id': 'MED0001', 'count': 2}]


In [24]:
# E.4 Ejecucion adaptativa: el plan inicial (antes de ejecutar) frente al plan final (despues). Dos lineas de bitacora
reg_aqe, plan_aqe, resultado_aqe = correr("C-aqe")
campos = ["consulta", "aqe", "etapas", "tareas", "hash_resultado", "hash_plan_inicial", "hash_plan_final", "estrategia_union"]
for r in (RESULTADOS["C"], reg_aqe):
    print(json.dumps({k: (r.get(k)[:12] if k.startswith("hash") and r.get(k) else r.get(k)) for k in campos}))
print("etapas omitidas por la ejecucion adaptativa:", reg_aqe["etapas_omitidas"])
# Con el umbral desactivado la ejecucion adaptativa NO convierte la mezcla en difusion
P.configurar(spark, aqe=True, umbral_difusion=-1)
r_menos1, _, _, _ = P.medir_df(spark, Q1["C"](capa, datos["referencia_df"], DIA))
P.configurar(spark)
print("adaptativa activa con umbral -1:", r_menos1["estrategia_union"], "· tareas", r_menos1["tareas"])

{"consulta": "C", "aqe": false, "etapas": 3, "tareas": 13, "hash_resultado": "cf38f78f6f2a", "hash_plan_inicial": null, "hash_plan_final": null, "estrategia_union": "BroadcastHashJoin"}
{"consulta": "C-aqe", "aqe": true, "etapas": 3, "tareas": 6, "hash_resultado": "cf38f78f6f2a", "hash_plan_inicial": "fe97c6731ebb", "hash_plan_final": "2c6a0d076b5d", "estrategia_union": "BroadcastHashJoin"}
etapas omitidas por la ejecucion adaptativa: 1


adaptativa activa con umbral -1: SortMergeJoin · tareas 9


In [25]:
# Autoverificacion del sub-paso E
c, f, q = RESULTADOS["C"], RESULTADOS["C-forzada"], RESULTADOS["C-aqe"]
assert c["estrategia_union"] == "BroadcastHashJoin" and c["n_exchange"] == 1
assert f["estrategia_union"] == "SortMergeJoin" and f["n_exchange"] == 3 and f["n_sort"] == 2
assert f["bytes_mezcla"] > 1000 * c["bytes_mezcla"]
assert c["hash_resultado"] == f["hash_resultado"] == q["hash_resultado"]                # el resultado no cambia
assert q["aqe"] is True and q["hash_plan_inicial"] != q["hash_plan_final"]
assert int(con_historia["filas"]) == int(en_capa["filas"]) + 2_880                        # la clave duplicada multiplica filas
assert r_menos1["estrategia_union"] == "SortMergeJoin"
print("Sub-paso E: OK")

Sub-paso E: OK


**Su decisión · Sub-paso E (decisión 4).** **Difundir** la referencia y dejar a Catalyst con el umbral por omisión. En la base, C difunde con **707 B** de mezcla, frente a **4 391 404 B** al forzar SortMergeJoin, con el mismo resultado. En la ampliación: 724 B frente a 7 799 575 B. El **tamaño estimado** del lado de la referencia fue de 1 639 B, pero el **tamaño real** de lo difundido (*data size* del `BroadcastExchange`) fue de 4 202 480 B. La estimación subestima unas 2 500 veces, porque se difunde la tabla hash construida y no el archivo de 3 KiB. Aun así cabe bajo los 10 MiB. Cambiaríamos la decisión si el tamaño real de la difusión se acercara al umbral (unos 870 medidores, extrapolando linealmente) o si usáramos la referencia **con historia** sin filtrar: un medidor repetido agregó 2 880 filas y 461,6 m³ de más.

### Tabla de planes del Nivel 1

La celda siguiente reúne lo medido en una tabla, la guarda como `planes_nivel1.csv` y guarda el plan normalizado de cada consulta en `planes_nivel1/`. Súbanlos al repositorio del equipo.

In [26]:
# Tabla de planes del Nivel 1: lo que midieron, listo para el repositorio del equipo
orden = ["A", "A-fecha", "B", "C", "C-forzada", "C-aqe"]
columnas = ["partition_filters", "pushed_filters", "read_schema", "archivos_leidos", "filas_salida_escaneo", "estrategia_union", "n_exchange"]
planes_n1 = pd.DataFrame([{"consulta": n, **{c: RESULTADOS[n][c] for c in columnas}} for n in orden])
planes_n1.to_csv(RAIZ / "planes_nivel1.csv", index=False)
for f in planes_n1.itertuples(index=False):          # dos lineas por consulta: lo que dice el plan y lo que se midio
    print(f"{f.consulta:9s} PartitionFilters: {f.partition_filters or '(ninguno)'}")
    print(f"{'':9s} PushedFilters: {f.pushed_filters or '(ninguno)'} · ReadSchema: {f.read_schema}")
    print(f"{'':9s} archivos {f.archivos_leidos} · filas de salida del escaneo {f.filas_salida_escaneo} · union {f.estrategia_union} · Exchange {f.n_exchange}")
(RAIZ / "planes_nivel1").mkdir(exist_ok=True)
for n in orden:                       # el plan normalizado de cada consulta, para versionarlo
    (RAIZ / "planes_nivel1" / f"plan_base_{n}.txt").write_text(PLANES[n], encoding="utf-8")
print("guardados: planes_nivel1.csv y planes_nivel1/plan_base_<consulta>.txt")

A         PartitionFilters: [isnotnull(anio), isnotnull(mes), isnotnull(dia), (anio = 2024), (mes = 02), (dia = 15)]
          PushedFilters: (ninguno) · ReadSchema: struct<sector:string,consumo_m3:double>
          archivos 1 · filas de salida del escaneo 4800 · union None · Exchange 1
A-fecha   PartitionFilters: (ninguno)
          PushedFilters: [IsNotNull(fecha), EqualTo(fecha,2024-02-15)] · ReadSchema: struct<sector:string,consumo_m3:double,fecha:date>
          archivos 120 · filas de salida del escaneo 4800 · union None · Exchange 1
B         PartitionFilters: (ninguno)
          PushedFilters: [IsNotNull(presion_bar), GreaterThan(presion_bar,3.5)] · ReadSchema: struct<sector:string,consumo_m3:double,presion_bar:double>
          archivos 120 · filas de salida del escaneo 576000 · union None · Exchange 1
C         PartitionFilters: (ninguno)
          PushedFilters: [IsNotNull(es_confiable), EqualTo(es_confiable,true), IsNotNull(meter_id)] · ReadSchema: struct<meter_id:string,co

In [27]:
# Cierre del Nivel 1: el lago heredado no cambio (la capa nueva y la bitacora no se cuentan)
assert P.hash_arbol(LAGO_BASE) == HASH_LAGO_ANTES
print("datalake/ identico al inicio:", HASH_LAGO_ANTES[:16], "...")

datalake/ identico al inicio: adac5febf8e7e3f5 ...


**Consolidación (por equipo).**

(1) **B** lee casi lo mismo con cualquier filtro sobre la presión: la presión alta está en todos los días (el máximo por archivo es de al menos 3,95 bar), así que ninguna estadística de archivo permite descartar uno. Lo mismo pasa con C: `es_confiable` está en todos los días.

(2) Si la historia se multiplicara por 10, cambiaría primero **`archivos_leidos` de A-fecha, B y C** (de 120 a 1 200) y sus `filas_salida_escaneo`. **No cambiaría `archivos_leidos` de A** (sigue siendo 1, por la poda) ni la estrategia de unión: la referencia depende de los medidores, no de los días.

(3) Para la dirección financiera: *"El costo del cierre estaba en cómo se pedía el día, no en el volumen de datos: pedido por la carpeta del día se lee un archivo; pedido por la fecha guardada adentro se abren todos los días de la historia, y ese costo crece cada día."*

## Nivel 2 · Aplicado · 75 minutos dentro de las 3 horas de la tarea

**Presupuesto (estimado).** Comprobar la ampliación y leer cómo está guardada su capa, 5 min; escribir las cuatro funciones y llevarlas al módulo, 20 min; hipótesis y ejecución de las seis consultas, 10 min; tabla de hipótesis frente a lo observado y lector independiente, 15 min; costo y diseño alternativo, 10 min; evidencia y verificación, 10 min; colchón, 5 min. El resto de las 3 horas es para el informe de los tres planes, el registro de decisiones, la sección de costo, el glosario y el índice. La ejecución ocupa de 3 a 5 minutos; lo que consume el tiempo es escribir y redactar.

> **Antes del Nivel 2.** La ampliación de su equipo, su capa refinada y su referencia ya deben existir: se generan con los tres comandos de "Antes de empezar" de la guía, **con este cuaderno cerrado o sin sesión de Spark abierta**. La celda N2.1 lo comprueba. Si reinician el núcleo, ejecuten de nuevo: la configuración común, los ayudantes, B.2, B.3 y C.0.

### El encargo

Repitan el patrón del Nivel 1 sobre **la ampliación de su equipo**, pero ahora las consultas son **suyas**: escriben `dia_de_a`, `consulta_a`, `consulta_b` y `consulta_c`, las llevan a `pipeline/planes_s17.py`, registran una hipótesis por consulta **antes** de medir y ejecutan las seis consultas con tres corridas medidas. Cambien `EQUIPO` en la celda de configuración (o definan la variable de entorno `EQUIPO`) y ejecuten en orden.

In [28]:
# N2.1 La ampliacion de su equipo, su capa y su referencia deben existir (seccion "Antes de empezar" de la guia)
P.configurar(spark)                                   # el Nivel 1 dejo la ejecucion adaptativa activa en su ultima consulta
necesarios = [LAGO_EQUIPO / "cruda", LAGO_EQUIPO / "refinada_spark", LAGO_EQUIPO / "totales_control.csv",
              LAGO_EQUIPO / "referencia" / "medidores.parquet", LAGO_EQUIPO / "referencia" / "sectores.parquet"]
faltan = [str(r.relative_to(RAIZ)) for r in necesarios if not r.exists()]
assert not faltan, f"Falta {faltan}: ejecuten primero los tres comandos de 'Antes de empezar' con el cuaderno cerrado."
perfil = json.loads((LAGO_EQUIPO / "parametros_equipo.json").read_text(encoding="utf-8"))
print({k: perfil[k] for k in ("equipo", "medidores", "sector_dominante", "fraccion_dominante") if k in perfil})
assert perfil["equipo"] == EQUIPO, "EQUIPO no coincide con la ampliacion generada"
HASH_EQUIPO_ANTES = P.hash_arbol(LAGO_EQUIPO)
print("hash de datalake_equipo/ al empezar:", HASH_EQUIPO_ANTES[:16], "...")

{'equipo': 'equipo-1', 'medidores': 348, 'sector_dominante': 'S-SUR', 'fraccion_dominante': 0.3448}


hash de datalake_equipo/ al empezar: 69fc81c464733edc ...


In [29]:
# N2.1b Como esta guardada SU capa: de ello dependen los archivos que cada consulta podra evitar
datos2 = P.preparar_datos(spark, LAGO_EQUIPO, dia=DIA)   # lee su capa y su referencia una sola vez (el dia de A se fija en N2.3)
archivos2 = sorted(datos2["capa"].rglob("*.parquet"))
N_ARCHIVOS = len(archivos2)
tam2 = [a.stat().st_size for a in archivos2]
md2 = pq.ParquetFile(archivos2[0]).metadata
lecturas2 = int(datos2["capa_df"].count())
print(f"{N_ARCHIVOS} archivos · {sum(tam2) / 1024:.1f} KiB en total · {sum(tam2) / N_ARCHIVOS / 1024:.2f} KiB por archivo · "
      f"{md2.num_row_groups} grupo de filas de {md2.num_rows} filas")
print("lecturas:", lecturas2, "· particiones de lectura de la capa completa:", datos2["capa_df"].rdd.getNumPartitions())

120 archivos · 4413.4 KiB en total · 36.78 KiB por archivo · 1 grupo de filas de 8352 filas
lecturas: 1002240 · particiones de lectura de la capa completa: 4


### N2.2 Su turno: cuatro funciones

Cada celda de ejercicio va seguida de su autoverificación sobre la base, que reproduce lo que ya hicieron en el Nivel 1. **Decisiones de ingeniería de este nivel:** qué día y qué predicado (`dia_de_a`, `consulta_a`), qué columnas leer y cómo unir (`consulta_b`, `consulta_c`). La estrategia de unión no se fija en `consulta_c`: la fija la configuración de la sesión (`ejecutar_consulta` pone el umbral en -1 para `C-forzada` y enciende la ejecución adaptativa para `C-aqe`).

**N2.2a · `dia_de_a`**

In [30]:
# N2.2a Su turno: el dia de la consulta A (el de mayor consumo total del equipo)
def dia_de_a(totales_control):
    """Dia de la consulta A.

    Encargo (resuelto por equipo-1): totales_control es un DataFrame de pandas con las columnas fecha (texto
    'AAAA-MM-DD'), sector, n_lecturas y consumo_total_m3: una fila por dia y sector, sin filtro de
    calidad. Devuelvan la fecha, como texto, del dia de MAYOR consumo total (la suma de los cuatro
    sectores). Si dos dias empatan, la fecha mas temprana.
    Anoten en su registro de decisiones por que esta regla da un dia distinto a cada equipo y que
    harian si no existiera un archivo de totales de control.

    Decision (equipo-1): cada ampliacion tiene su propio volumen y sector dominante, asi que el dia
    de mayor consumo total cambia de un equipo a otro. Sin totales de control, lo calculariamos con
    un groupBy(fecha) sobre la propia capa (una pasada completa) y lo fijariamos con --dia."""
    por_dia = totales_control.groupby("fecha", as_index=False)["consumo_total_m3"].sum()
    por_dia = por_dia.sort_values(["consumo_total_m3", "fecha"], ascending=[False, True], kind="mergesort")
    return str(por_dia.iloc[0]["fecha"])


In [31]:
# Autoverificacion de dia_de_a: un caso con empate y el dia real del equipo
sintetico = pd.DataFrame({"fecha": ["2024-01-02", "2024-01-01", "2024-01-02", "2024-01-01"], "sector": ["S-NORTE", "S-NORTE", "S-SUR", "S-SUR"],
                          "consumo_total_m3": [5.0, 6.0, 5.0, 4.0]})
assert dia_de_a(sintetico) == "2024-01-01"            # empate (10 y 10): gana la fecha mas temprana
totales = pd.read_csv(LAGO_EQUIPO / "totales_control.csv")
DIA_A = dia_de_a(totales)
assert isinstance(DIA_A, str) and len(DIA_A) == 10 and DIA_A in set(totales["fecha"])
print("Dia de la consulta A de", EQUIPO, ":", DIA_A, "· consumo de ese dia", round(float(totales[totales["fecha"] == DIA_A]["consumo_total_m3"].sum()), 3), "m3")

Dia de la consulta A de equipo-1 : 2024-01-14 · consumo de ese dia 1343.381 m3


**N2.2b · `consulta_a`**

In [32]:
# N2.2b Su turno: consulta A, consumo por sector de un dia filtrando por las columnas de la ruta
from pyspark.sql import functions as F

def consulta_a(capa, dia):
    """Consumo por sector de UN dia, filtrando por las columnas de la ruta.

    Encargo (resuelto por equipo-1): capa es el DataFrame de refinada_spark/ y dia es un texto 'AAAA-MM-DD'.
      - Filtren por anio, mes y dia. Esas tres columnas se leen como TEXTO con ceros ('2024',
        '02', '15'): el literal debe coincidir en tipo y en ceros con la columna.
      - No usen la columna fecha: esta guardada DENTRO de los archivos y no permite podar.
      - Agrupen por sector y devuelvan dos columnas: n_lecturas (conteo) y consumo_total_m3
        (suma de consumo_m3)."""
    anio, mes, d = dia.split("-")
    return (capa.filter((F.col("anio") == F.lit(anio)) & (F.col("mes") == F.lit(mes)) & (F.col("dia") == F.lit(d)))
                .groupBy("sector").agg(F.count("*").alias("n_lecturas"), F.sum("consumo_m3").alias("consumo_total_m3")))


In [33]:
# Autoverificacion de consulta_a sobre la BASE (el mismo dia del Nivel 1)
r, p, _, res = P.medir_df(spark, consulta_a(capa, DIA))
assert int(res["n_lecturas"].sum()) == 4_800 and len(res) == 4
assert r["archivos_leidos"] == 1 and all(c in r["partition_filters"] for c in ("anio", "mes", "dia"))
assert set(res.columns) == {"sector", "n_lecturas", "consumo_total_m3"}
print("consulta_a: OK · lee", r["archivos_leidos"], "archivo de 120")

consulta_a: OK · lee 1 archivo de 120


**N2.2c · `consulta_b`**

In [34]:
# N2.2c Su turno: consulta B, lecturas con presion alta por sector
UMBRAL_PRESION = 3.5

def consulta_b(capa, umbral=3.5):
    """Lecturas con presion alta por sector.

    Encargo (resuelto por equipo-1): devuelvan, por sector, n_lecturas y consumo_total_m3 de las lecturas cuya
    presion_bar es MAYOR que `umbral` (3,5 bar por omision). La presion es una columna sin
    particion ni orden: anticipen en su hipotesis cuantos archivos se leeran y por que."""
    return (capa.filter(F.col("presion_bar") > F.lit(umbral))
                .groupBy("sector").agg(F.count("*").alias("n_lecturas"), F.sum("consumo_m3").alias("consumo_total_m3")))


In [35]:
# Autoverificacion de consulta_b sobre la BASE
r, p, _, res = P.medir_df(spark, consulta_b(capa))
assert int(res["n_lecturas"].sum()) == 27_472
assert r["partition_filters"] == "" and "presion_bar" in r["pushed_filters"] and r["archivos_leidos"] == 120
print("consulta_b: OK ·", int(res["n_lecturas"].sum()), "lecturas con presion alta")

consulta_b: OK · 27472 lecturas con presion alta


**N2.2d · `consulta_c`**

In [36]:
# N2.2d Su turno: consulta C, consumo por estrato de las lecturas confiables, uniendo con la referencia
def consulta_c(capa, medidores):
    """Consumo por estrato de las lecturas confiables: union con la referencia de medidores.

    Encargo (resuelto por equipo-1): capa trae las lecturas y medidores la referencia (meter_id, sector, estrato,
    fecha_inicio, fecha_fin, es_actual). Se quiere, por estrato, n_lecturas y consumo_total_m3 de
    las lecturas con es_confiable verdadero.
      - Lleven a la union solo las columnas que hacen falta de cada lado.
      - Unan por meter_id. No fijen ninguna pista de union ni cambien el umbral aqui: la
        estrategia la fija la configuracion de la sesion (ver configurar y ejecutar_consulta).
      - Antes de ejecutar, escriban cual esperan que sea la estrategia de union."""
    lecturas = capa.filter(F.col("es_confiable")).select("meter_id", "consumo_m3")
    ref = medidores.select("meter_id", "estrato")
    return (lecturas.join(ref, on="meter_id", how="inner")
                    .groupBy("estrato").agg(F.count("*").alias("n_lecturas"), F.sum("consumo_m3").alias("consumo_total_m3")))


In [37]:
# Autoverificacion de consulta_c sobre la BASE
r, p, _, res = P.medir_df(spark, consulta_c(capa, datos["referencia_df"]))
assert int(res["n_lecturas"].sum()) == 541_368 and sorted(res["estrato"]) == [1, 2, 3, 4]
assert r["estrategia_union"] == "BroadcastHashJoin" and r["archivos_leidos"] == 120
print("consulta_c: OK · estrategia por omision:", r["estrategia_union"])

consulta_c: OK · estrategia por omision: BroadcastHashJoin


### N2.3 Las cuatro funciones, en el módulo

Cuando las cuatro autoverificaciones pasen, **lleven las funciones a `pipeline/planes_s17.py`** (explorador de archivos de JupyterLab, carpeta `pipeline/`): reemplacen cada función pendiente **entera, con su docstring incluido**, por la suya y guarden (si queda un `TODO` o un `NotImplementedError` dentro de la función, la celda lo detecta). Después ejecuten la celda siguiente, que recarga el módulo, comprueba que no queda ninguna pendiente y fija el día de A con **su** `dia_de_a`.

In [38]:
# N2.3 Las cuatro funciones, en el modulo: recarguen planes_s17.py y comprueben que ya no queda ninguna pendiente
# (ni NotImplementedError ni TODO: se reemplaza la funcion ENTERA, docstring incluida)
importlib.reload(P)
pendientes = [n for n in ("dia_de_a", "consulta_a", "consulta_b", "consulta_c")
              if any(m in inspect.getsource(getattr(P, n)) for m in ("NotImplementedError", "TODO"))]
assert not pendientes, f"Falta llevar al modulo: {pendientes}"
datos2["dia"] = P.dia_por_omision(LAGO_EQUIPO)         # el modulo usa SU dia_de_a para fijar el dia de A
assert datos2["dia"] == DIA_A
print("modulo recargado · dia de A:", datos2["dia"], "· archivos de la capa:", N_ARCHIVOS)

modulo recargado · dia de A: 2024-01-14 · archivos de la capa: 120


### N2.4 Hipótesis antes de medir

Para cada consulta, escriban cuántos archivos esperan que lea y, en C, qué estrategia de unión. La celda siguiente guarda las hipótesis en `hipotesis_s17.json` **con una marca de tiempo**, y la verificación exige que esa marca sea anterior a la de la medición. Una hipótesis equivocada no baja la calificación; una hipótesis escrita después de medir no vale.

In [39]:
# N2.4a Su turno: la hipotesis de cada consulta, escrita ANTES de medir
# equipo-1: la capa de la ampliacion tiene 120 archivos (uno por dia, de unos 38 KiB) y la referencia de medidores
# es una tabla de 348 filas (unos KiB), muy por debajo del umbral de difusion de 10 MiB.
HIPOTESIS = {
    "A":         {"texto": "Filtra por anio, mes y dia, que son columnas de la RUTA: PartitionFilters poda 119 de las 120 carpetas y se abre solo el archivo del dia.",
                  "archivos_leidos": 1, "estrategia_union": None},
    "A-fecha":   {"texto": "fecha esta DENTRO del archivo, no en la ruta: no hay PartitionFilters, el filtro solo se empuja al lector (PushedFilters), asi que se abren los 120 archivos para mirar sus estadisticas, aunque devuelva las mismas filas que A.",
                  "archivos_leidos": 120, "estrategia_union": None},
    "B":         {"texto": "presion_bar no es columna de particion y no esta ordenada: cada archivo diario tiene presiones por encima y por debajo de 3,5 bar (max > 3,5), asi que el predicado empujado no puede descartar ningun archivo: 120.",
                  "archivos_leidos": 120, "estrategia_union": None},
    "C":         {"texto": "Lee toda la capa (es_confiable no es particion): 120 archivos. La referencia de medidores (348 filas) estimada en KiB queda bajo el umbral de 10 MiB: Catalyst la difunde (BroadcastHashJoin) y no mezcla las lecturas.",
                  "archivos_leidos": 120, "estrategia_union": "BroadcastHashJoin"},
    "C-forzada": {"texto": "Mismos 120 archivos; con el umbral en -1 no puede difundir: repartira ambos lados por meter_id y los ordenara (SortMergeJoin), con al menos 2 Sort y 3 Exchange y muchos mas bytes de mezcla que C.",
                  "archivos_leidos": 120, "estrategia_union": "SortMergeJoin"},
    "C-aqe":     {"texto": "Mismos 120 archivos; con ejecucion adaptativa el plan inicial ya difunde la referencia pequena, asi que el plan final tambien sera BroadcastHashJoin (AQE solo coalesce las particiones de la agregacion).",
                  "archivos_leidos": 120, "estrategia_union": "BroadcastHashJoin"},
}


In [40]:
# N2.4b Registrar las hipotesis: la marca de tiempo queda ANTES de cualquier medicion
assert all(isinstance(h["archivos_leidos"], int) and h["texto"] != "TODO" for h in HIPOTESIS.values()), "Completen las seis hipotesis"
P.registrar_hipotesis(RAIZ / "hipotesis_s17.json", HIPOTESIS)
print("hipotesis registradas:", {k: (v["archivos_leidos"], v["estrategia_union"]) for k, v in HIPOTESIS.items()})
print("marca de tiempo:", HIPOTESIS["A"]["marca_hipotesis"])

hipotesis registradas: {'A': (1, None), 'A-fecha': (120, None), 'B': (120, None), 'C': (120, 'BroadcastHashJoin'), 'C-forzada': (120, 'SortMergeJoin'), 'C-aqe': (120, 'BroadcastHashJoin')}
marca de tiempo: 2026-11-12T22:17:34.046087Z


### N2.5 Las seis consultas

`ejecutar_todo` ejecuta cada consulta con la configuración que le corresponde: una corrida de calentamiento (se descarta) y tres medidas, con el plan normalizado, las métricas del escaneo y de las etapas, el resultado idéntico entre variantes y el cálculo independiente con pandas y pyarrow. Anexa 18 líneas a `bitacora_planes.jsonl` y guarda un plan normalizado por consulta en `planes/`. La ejecución ocupa unos tres a cuatro minutos: es buen momento para redactar el informe.

In [41]:
# N2.5 Las seis consultas sobre SU ampliacion: 1 corrida de calentamiento (se descarta) y 3 medidas por consulta,
# con calculo independiente en pandas y pyarrow. Anexa 18 lineas a bitacora_planes.jsonl y guarda 6 planes en planes/
t0 = time.perf_counter()
ultimas = P.ejecutar_todo(spark, datos2, HIPOTESIS, EQUIPO, "ampliacion", 3, RAIZ / "bitacora_planes.jsonl", RAIZ / "planes")
P.configurar(spark)
print(f"seis consultas en {time.perf_counter() - t0:.0f} s")
for n, r in ultimas.items():
    print(f"{n:10s} archivos {r['archivos_leidos']:3d} · filas del escaneo {r['filas_salida_escaneo']:8d} · union {str(r['estrategia_union']):17s} "
          f"· Exchange {r['n_exchange']} · etapas {r['etapas']} · tareas {r['tareas']:3d} · mezcla {r['bytes_mezcla']} B")

seis consultas en 13 s
A          archivos   1 · filas del escaneo     8352 · union None              · Exchange 1 · etapas 2 · tareas   9 · mezcla 224 B
A-fecha    archivos 120 · filas del escaneo     8352 · union None              · Exchange 1 · etapas 2 · tareas  12 · mezcla 224 B
B          archivos 120 · filas del escaneo  1002240 · union None              · Exchange 1 · etapas 2 · tareas  12 · mezcla 891 B
C          archivos 120 · filas del escaneo  1002240 · union BroadcastHashJoin · Exchange 1 · etapas 3 · tareas  13 · mezcla 724 B
C-forzada  archivos 120 · filas del escaneo  1002240 · union SortMergeJoin     · Exchange 3 · etapas 4 · tareas  21 · mezcla 7799575 B
C-aqe      archivos 120 · filas del escaneo  1002240 · union BroadcastHashJoin · Exchange 1 · etapas 3 · tareas   6 · mezcla 724 B


### N2.6 Hipótesis frente a lo observado

Para cada consulta: ¿se cumplió la hipótesis? Si no, ¿por qué? La diferencia explicada es la evidencia.

In [42]:
# N2.6 Hipotesis frente a lo observado, por consulta (ultima corrida medida de la bitacora)
bitacora = [json.loads(l) for l in (RAIZ / "bitacora_planes.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
mias = [b for b in bitacora if b["equipo"] == EQUIPO and b["volumen"] == "ampliacion"]
ultimas_b = {b["consulta"]: b for b in mias}               # la ultima linea de cada consulta
tabla_obs = pd.DataFrame([{"consulta": n, "archivos esperados": r["hipotesis_archivos_leidos"], "archivos leidos": r["archivos_leidos"],
                           "filas de salida del escaneo": r["filas_salida_escaneo"], "union esperada": r["hipotesis_estrategia_union"],
                           "union observada": r["estrategia_union"], "Exchange": r["n_exchange"], "etapas": r["etapas"], "tareas": r["tareas"],
                           "bytes de mezcla": r["bytes_mezcla"], "diferencia max. con el calculo independiente": f"{r['diferencia_max_independiente']:.1e}"}
                          for n, r in ultimas_b.items()])
print(tabla_obs.to_string(index=False))
print("lineas de la bitacora de este equipo:", len(mias), "· corridas por consulta:", sorted({b["corrida"] for b in mias}))
print("hipotesis cumplidas:", {n: r["hipotesis_archivos_leidos"] == r["archivos_leidos"] for n, r in ultimas_b.items()})

 consulta  archivos esperados  archivos leidos  filas de salida del escaneo    union esperada   union observada  Exchange  etapas  tareas  bytes de mezcla diferencia max. con el calculo independiente
        A                   1                1                         8352              None              None         1       2       9              224                                      9.1e-13
  A-fecha                 120              120                         8352              None              None         1       2      12              224                                      9.1e-13
        B                 120              120                      1002240              None              None         1       2      12              891                                      1.8e-12
        C                 120              120                      1002240 BroadcastHashJoin BroadcastHashJoin         1       3      13              724                                      7.3e-10


### N2.7 El lector independiente sobre su capa

Las estadísticas de Parquet explican el plan B: ¿cuántos archivos podría saltar un umbral de 4,0 o de 4,2 bar en **su** ampliación? ¿Cuánto pesan, en bytes comprimidos, las columnas que lee cada consulta?

In [43]:
# N2.7 Lector independiente sobre SU capa: por que B no ahorra archivos y cuanto pesa cada columna
est2, col2 = P.estadisticas_parquet(datos2["capa"])
print("filas por grupo:", sorted(est2["filas"].unique()), "· grupos de filas por archivo:", sorted(est2["grupos_de_filas"].unique()))
print("menor de los maximos de presion_bar:", round(est2["presion_bar_max"].min(), 3))
for umbral in (3.5, 4.0, 4.2):
    print(f"  archivos saltables con {umbral} bar: {int((est2['presion_bar_max'] <= umbral).sum())} de {len(est2)}")
f2 = col2.set_index("columna")["fraccion"]
print(f"fraccion de bytes: A {f2[['sector', 'consumo_m3']].sum():.1%} · B {f2[['sector', 'consumo_m3', 'presion_bar']].sum():.1%} · "
      f"C {f2[['meter_id', 'consumo_m3', 'es_confiable']].sum():.1%}")

filas por grupo: [8352] · grupos de filas por archivo: [1]
menor de los maximos de presion_bar: 3.977
  archivos saltables con 3.5 bar: 0 de 120
  archivos saltables con 4.0 bar: 3 de 120
  archivos saltables con 4.2 bar: 97 de 120
fraccion de bytes: A 29.1% · B 78.5% · C 38.5%


### N2.8 Costo de leer y punto de equilibrio

**a.** Se escribe la capa **por mes** en una carpeta aparte (`_diseno/`) y se miden el cierre del día y el reporte de ese mes con los dos diseños. **b.** Con sus cifras, se proyectan los archivos y bytes de un año de cierres diarios (historia inicial de 120 días) y se calcula el punto de equilibrio $n^{*}=C_{\text{cambio}}/(c_{\text{actual}}-c_{\text{nuevo}})$ de cada cambio, que no está definido si el cambio no ahorra nada. Ningún precio por unidad es dato: si lo usan, es un supuesto suyo.

In [44]:
# N2.8a Un diseno alternativo: la capa por MES (un archivo por mes). Se escribe aparte (_diseno/) y se mide el mismo
# cierre diario y el reporte de ese mes, para calcular si cambiar de diseno se paga.
destino_mes = RAIZ / "_diseno" / "por_mes"
spark.conf.set("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS")
t0 = time.perf_counter()
datos2["capa_df"].repartition("anio", "mes").write.mode("overwrite").partitionBy("anio", "mes").parquet(str(destino_mes))
capa_mes = P.leer_capa(spark, destino_mes)
anio_a, mes_a, _ = DIA_A.split("-")
dias_del_mes = int(datos2["capa_df"].filter((F.col("anio") == anio_a) & (F.col("mes") == mes_a)).select("dia").distinct().count())
medidas_mes = {
    "por dia · cierre por la ruta": P.medir_df(spark, agregar(datos2["capa_df"].filter((F.col("anio") == anio_a) & (F.col("mes") == mes_a) & (F.col("dia") == DIA_A[8:]))))[0],
    "por mes · cierre del dia": P.medir_df(spark, agregar(capa_mes.filter((F.col("anio") == anio_a) & (F.col("mes") == mes_a) & (F.col("fecha") == F.lit(DIA_A).cast("date")))))[0],
    "por dia · reporte del mes": P.medir_df(spark, agregar(datos2["capa_df"].filter((F.col("anio") == anio_a) & (F.col("mes") == mes_a))))[0],
    "por mes · reporte del mes": P.medir_df(spark, agregar(capa_mes.filter((F.col("anio") == anio_a) & (F.col("mes") == mes_a))))[0],
}
archivos_mes = sorted(destino_mes.rglob("*.parquet"))
print(f"el mes de {DIA_A[:7]} tiene {dias_del_mes} dias cargados")
print(f"capa por mes: {len(archivos_mes)} archivos · {sum(a.stat().st_size for a in archivos_mes) / 1024:.1f} KiB (la capa por dia pesa {sum(tam2) / 1024:.1f} KiB) · escrita en {time.perf_counter() - t0:.0f} s")
print(pd.DataFrame([{"diseno y consulta": k, "archivos": r["archivos_leidos"], "KiB leidos": round(r["tamano_archivos_leidos_b"] / 1024, 1),
                     "filas de salida del escaneo": r["filas_salida_escaneo"]} for k, r in medidas_mes.items()]).to_string(index=False))

el mes de 2024-01 tiene 31 dias cargados
capa por mes: 4 archivos · 3169.2 KiB (la capa por dia pesa 4413.4 KiB) · escrita en 3 s
           diseno y consulta  archivos  KiB leidos  filas de salida del escaneo
por dia · cierre por la ruta         1        36.7                         8352
    por mes · cierre del dia         1       818.2                       198912
   por dia · reporte del mes        31      1140.4                       258912
   por mes · reporte del mes         1       818.2                       258912


In [45]:
# N2.8b Costo de leer: proyeccion a un ano de cierres diarios y punto de equilibrio, con SUS cifras
def punto_de_equilibrio(costo_cambio, costo_actual, costo_nuevo):
    """Cierres necesarios para recuperar el costo de un cambio; infinito si el cambio no ahorra nada."""
    return costo_cambio / (costo_actual - costo_nuevo) if costo_actual > costo_nuevo else float("inf")

H, D = 120, 365                                            # historia inicial de 120 dias y un ano de cierres
kib_dia = ultimas_b["A"]["tamano_archivos_leidos_b"] / 1024    # tamano de UN archivo diario (el que lee A)
acumulado = sum(range(H + 1, H + D + 1))                   # archivos-dia leidos en D cierres sin poda
costo = pd.DataFrame([
    {"diseno y filtro": "por dia, filtro por la ruta (poda)", "archivos por cierre": 1, "KiB por cierre": round(kib_dia, 1), "archivos en 365 cierres": D, "MiB en 365 cierres": round(D * kib_dia / 1024, 1)},
    {"diseno y filtro": "por dia, filtro por fecha (sin poda)", "archivos por cierre": round(acumulado / D), "KiB por cierre": round(acumulado / D * kib_dia, 1), "archivos en 365 cierres": acumulado, "MiB en 365 cierres": round(acumulado * kib_dia / 1024, 1)},
])
print(costo.to_string(index=False))
print(f"razon sin poda / con poda: {(2 * H + D + 1) / 2:.0f}")
# Cambio 1: corregir el filtro (no se reescribe ningun dato)
print("corregir el filtro: ahorra", round(acumulado / D) - 1, "archivos por cierre · punto de equilibrio", punto_de_equilibrio(0, acumulado / D, 1), "cierres")
# Cambio 2: pasar a diseno por mes. Cuesta reescribir la historia: archivos leidos mas archivos escritos
tocados = N_ARCHIVOS + len(archivos_mes)
cierre_dia = medidas_mes["por dia · cierre por la ruta"]["archivos_leidos"]
cierre_mes = medidas_mes["por mes · cierre del dia"]["archivos_leidos"]
kib_cierre_dia = medidas_mes["por dia · cierre por la ruta"]["tamano_archivos_leidos_b"] / 1024
kib_cierre_mes = medidas_mes["por mes · cierre del dia"]["tamano_archivos_leidos_b"] / 1024
print(f"cierre diario por mes: {cierre_mes} archivo frente a {cierre_dia} pero {kib_cierre_mes:.1f} KiB frente a {kib_cierre_dia:.1f} KiB -> "
      f"punto de equilibrio {punto_de_equilibrio(tocados, kib_cierre_dia, kib_cierre_mes)} (infinito: el cambio cuesta mas bytes y no se paga)")
ahorro_reporte = medidas_mes["por dia · reporte del mes"]["archivos_leidos"] - medidas_mes["por mes · reporte del mes"]["archivos_leidos"]
print(f"reporte del mes por mes: ahorra {ahorro_reporte} archivos por reporte · reescribir toca {tocados} archivos -> "
      f"{punto_de_equilibrio(tocados, medidas_mes['por dia · reporte del mes']['archivos_leidos'], medidas_mes['por mes · reporte del mes']['archivos_leidos']):.1f} reportes")

                     diseno y filtro  archivos por cierre  KiB por cierre  archivos en 365 cierres  MiB en 365 cierres
  por dia, filtro por la ruta (poda)                    1            36.7                      365                13.1
por dia, filtro por fecha (sin poda)                  303         11127.3                   110595              3966.3
razon sin poda / con poda: 303
corregir el filtro: ahorra 302 archivos por cierre · punto de equilibrio 0.0 cierres
cierre diario por mes: 1 archivo frente a 1 pero 818.2 KiB frente a 36.7 KiB -> punto de equilibrio inf (infinito: el cambio cuesta mas bytes y no se paga)
reporte del mes por mes: ahorra 30 archivos por reporte · reescribir toca 124 archivos -> 4.1 reportes


### N2.9 Evidencia y registro de decisiones

La celda genera `evidencia_nivel2_<equipo>.md` con el perfil, las hipótesis y lo observado; ustedes completan el contraste, las decisiones y la conclusión **separando lo observado (con su cifra) de lo interpretado (con su causa)**. El registro de decisiones usa el formato de la plantilla `plantillas/t17/decisiones.md` y justifica con cifras de **su** bitácora al menos tres decisiones, entre ellas la 1 (diseño físico) y la 4 (unión). Al entregar, versionen también `docker-compose.yml`, `perfiles/s15/spark/` (`Dockerfile` y `requirements.txt`, sin los cuales un clon limpio no construye la imagen) y este cuaderno **ejecutado**, con sus salidas.

In [46]:
# N2.9 evidencia_nivel2_<equipo>.md: el perfil, las hipotesis, las tablas y el espacio para las conclusiones
conclusion = ("OBSERVADO: A lee 1 archivo (PartitionFilters sobre anio, mes y dia) y A-fecha 120, con las mismas 8 352 filas y el mismo "
              "hash; B empuja presion_bar > 3,5 y aun asi lee 120 archivos y 1 002 240 filas; C difunde la referencia (724 B de mezcla) y "
              "C-forzada mezcla 7 799 575 B con 3 Exchange y 2 Sort, con el mismo resultado. INTERPRETADO: A poda porque el filtro esta en "
              "las columnas de la RUTA; B no ahorra archivos porque el menor de los maximos de presion por archivo es 3,977 bar, asi que "
              "ninguna estadistica permite saltar un archivo con el umbral de 3,5; la mezcla forzada cuesta lo que cuesta porque obliga a "
              "repartir y ordenar el millon de lecturas por meter_id en lugar de copiar 348 medidores a cada tarea.")
texto = f"""# Evidencia del Nivel 2 · {EQUIPO}

## 1. Perfil y configuracion
- Equipo: {EQUIPO} · volumen: ampliacion · medidores: {perfil.get('medidores')} · sector dominante: {perfil.get('sector_dominante')} ({perfil.get('fraccion_dominante')})
- Dia de la consulta A: {DIA_A} (el de mayor consumo total segun totales_control.csv)
- Spark {spark.version} · modo de explain: {P.MODO_EXPLAIN} · ejecucion adaptativa apagada salvo en C-aqe · particiones de mezcla: 8 · umbral de difusion: 10 MiB (C-forzada: -1) · columnas de particion como texto

## 2. Hipotesis (con su marca de tiempo)
{a_markdown(pd.DataFrame([{'consulta': k, 'archivos esperados': v['archivos_leidos'], 'union esperada': v['estrategia_union'], 'marca': v['marca_hipotesis'], 'razon': v['texto']} for k, v in HIPOTESIS.items()]))}

## 3. Observado
{a_markdown(tabla_obs)}

## 4. Contraste de cada hipotesis con lo observado
- A: se cumplio (1 archivo esperado, 1 leido): el filtro por la ruta poda 119 de 120 carpetas.
- A-fecha: se cumplio (120 y 120): fecha esta dentro del archivo; el lector salta grupos de filas pero abre los 120 archivos.
- B: se cumplio (120 y 120): el predicado se empuja pero ningun archivo tiene su maximo de presion por debajo de 3,5 bar.
- C: se cumplio (120 y 120, BroadcastHashJoin): la referencia (348 medidores) cabe bajo el umbral de 10 MiB.
- C-forzada: se cumplio (120 y 120, SortMergeJoin): con el umbral en -1, 3 Exchange y 2 Sort.
- C-aqe: se cumplio (120 y 120, BroadcastHashJoin): el plan final difunde igual; AQE reduce las tareas de 13 a 6.

## 5. Decisiones del nivel
- Dia de A: 2024-01-14, el de mayor consumo total (1 343,381 m3), con desempate por la fecha mas temprana; predicado sobre anio, mes y dia como texto con ceros. Lo cambiaria un archivos_leidos de A mayor que 1.
- Union: Catalyst con el umbral por omision (10 MiB), sin pista y con la referencia vigente. La cambiaria que difusion_bytes (hoy 4 210 672 B) se acerque al umbral.
- Diseno fisico: particion por dia (A lee 36,7 KiB por cierre frente a 818,2 KiB con el diseno por mes, cuyo punto de equilibrio es infinito para el cierre).

## 6. Conclusion
{conclusion}
"""
ruta_evidencia = RAIZ / f"evidencia_nivel2_{EQUIPO}.md"
ruta_evidencia.write_text(texto, encoding="utf-8")
print("escrito:", ruta_evidencia.name, f"({len(texto)} caracteres)")

escrito: evidencia_nivel2_equipo-1.md (4909 caracteres)


### N2.10 Verificación por etapa

`pipeline/verificar_etapa.py --etapa t17` lee archivos (no abre Spark), dice el valor observado frente al esperado y termina con un veredicto. Las dos últimas comprobaciones (entregables de texto y glosario) pasan cuando copian y completan las plantillas de `plantillas/t17/` en `entregables/t17/` y agregan la fila de T17 al índice de evidencias. La comprobación 16 verifica el lago intacto (`datalake_equipo/` sin las capas reconstruibles; con `--volumen base`, `datalake/`): los hashes `hash_datalake_antes` y `hash_datalake_despues` están solo en la línea de la consulta A de `bitacora_planes.jsonl`, y su mensaje impreso dice "datalake/ sin cambios".

In [47]:
# N2.10 Verificacion por etapa: lee archivos (no abre Spark) y dice el valor observado frente al esperado
r = subprocess.run([sys.executable, str(RAIZ / "pipeline" / "verificar_etapa.py"), "--etapa", "t17", "--raiz", str(RAIZ)], capture_output=True, text=True)
salida_t17 = r.stdout
aprobadas = salida_t17.count("[APROBADO")
for linea in salida_t17.splitlines():
    if linea.startswith("[NO APROBADO") or linea.startswith("Etapa"):
        print(linea[:200])
print("comprobaciones aprobadas:", aprobadas, "de 18")

Etapa t17: APROBADA (18 de 18 comprobaciones, 0.1 s)
comprobaciones aprobadas: 18 de 18


In [48]:
# Autoverificacion del Nivel 2: las 16 comprobaciones que dependen del codigo y la bitacora (las dos restantes
# comprueban los entregables de texto: informe, decisiones, costo, glosario e indice, que se redactan aparte)
assert aprobadas >= 16, "Alguna comprobacion de planes, hipotesis o resultado identico no pasa: lean la linea NO APROBADO"
assert P.hash_arbol(LAGO_EQUIPO) == HASH_EQUIPO_ANTES and P.hash_arbol(LAGO_BASE) == HASH_LAGO_ANTES    # el lago no cambio
assert (RAIZ / "planes" / "plan_ampliacion_C-aqe.txt").exists()
print("Nivel 2: OK")

Nivel 2: OK


## Nivel 3 · Frontera · opcional

**La decisión (sin guía).** Se une cada lectura con `sectores.parquet` por la clave `sector`, que tiene solo cuatro valores. Con 8 particiones de mezcla, ¿cuántas tendrán datos y cuál carga más filas? El reparto lo decide `hash(sector)` módulo 8, es decir, **el texto del sector**, no cuántas lecturas tenga. La celda N3.1 muestra a qué partición va cada sector; ustedes **predicen** la fracción de las filas que recibirá la partición más cargada, antes de medir, y comparan tres alternativas: **dejar a Catalyst** (que difunde la tabla de 4 filas y no mezcla la unión), **forzar la mezcla** (donde aparece la tarea pesada) y **mezclar con un sufijo aleatorio** en la clave. La ejecución adaptativa no es una opción aquí: solo reparte claves sesgadas con particiones de más de 256 MiB y cinco veces la mediana, que estos datos nunca alcanzan. Escriban la decisión 7 de su registro (alternativa adoptada, con su plan y sus cifras).

| Decisión | Opciones | Criterios para decidir | Consecuencia de decidir mal | Qué debe justificarse |
|---|---|---|---|---|
| 7 · Unión con clave sesgada | Dejar a Catalyst (difunde la referencia) · mezcla forzada (umbral -1) · mezcla con sufijo aleatorio (K = 8) | Fracción de las filas de la tarea más pesada (predicha y medida), bytes de mezcla, tamaño de la referencia frente a la memoria | Una tarea carga más de las tres cuartas partes de las filas, o se paga entre un 22 % y un 26 % más de bytes de mezcla (según el equipo) sin necesidad | Fracción predicha y medida, bytes de mezcla de cada alternativa, resultado idéntico y la cifra que haría cambiar la decisión |

**Por qué el sufijo no llega al reparto ideal.** Con el sufijo la clave pasa a 4 × 8 = 32 combinaciones que se reparten entre 8 particiones por el módulo de un hash: es un reparto al azar, y por azar algunas particiones reciben varias combinaciones y otras ninguna (la más cargada recibe entre el 25 % y el 30 % de las filas según el equipo, 26,6 % en equipo-3, no el 12,5 % ideal).

In [49]:
# N3.1 Union por la clave sector con sectores.parquet (4 filas). Con 4 claves y 8 particiones de mezcla, cuantas
# particiones tendran datos? El reparto lo decide hash(sector) modulo 8, es decir, el TEXTO del sector.
sectores = spark.read.parquet(str(LAGO_EQUIPO / "referencia" / "sectores.parquet")).select("sector", "estrato")
capa2 = datos2["capa_df"]
conteos = capa2.groupBy("sector").count().toPandas().set_index("sector")["count"]
mapa = P.particion_de_mezcla(spark, list(conteos.index))
print(pd.DataFrame({"lecturas": conteos, "particion de mezcla": pd.Series(mapa)}).sort_values("particion de mezcla").to_string())
print("sector dominante:", conteos.idxmax(), "con", f"{conteos.max() / conteos.sum():.1%}", "de las lecturas")

         lecturas  particion de mezcla
S-ESTE     218880                    2
S-NORTE    218880                    7
S-SUR      345600                    7
S-OESTE    218880                    7
sector dominante: S-SUR con 34.5% de las lecturas


In [50]:
# N3.2 Su turno: predecir, ANTES de medir, que fraccion de las filas llegara a la particion de mezcla mas cargada
def fraccion_predicha(conteos, mapa):
    """Suma las lecturas de los sectores que caen en la misma particion de mezcla (mapa: sector -> particion) y
    devuelve la fraccion del total que recibe la particion mas cargada."""
    por_particion = {}
    for sector, n in conteos.items():
        por_particion[mapa[sector]] = por_particion.get(mapa[sector], 0) + int(n)
    return max(por_particion.values()) / float(sum(int(n) for n in conteos.values))


In [51]:
# N3.3 Tres alternativas para la misma union: Catalyst por omision, mezcla forzada y mezcla con sufijo aleatorio (K = 8)
PREDICHA = fraccion_predicha(conteos, mapa)
alternativas = {"Catalyst por omision": ("catalyst", None), "Mezcla forzada (umbral -1)": ("catalyst", -1), "Sufijo aleatorio (K=8) con mezcla": ("sal", -1)}
filas, resultados3 = [], {}
for nombre, (estrategia, umbral) in alternativas.items():
    P.configurar(spark, umbral_difusion=umbral)
    df = P.consulta_sesgo(capa2, sectores, estrategia)
    marca = P.marca_interfaz(spark)
    pdf = df.toPandas()
    met = P.metricas_etapas(spark, marca)
    dist = P.distribucion_de_tareas(spark, marca)
    union = P.nodos_del_plan(P.normalizar_plan(P.capturar_plan(df)))["estrategia_union"]
    hay_mezcla = union != "BroadcastHashJoin"
    resultados3[nombre] = pdf.sort_values("estrato").reset_index(drop=True)
    filas.append({"alternativa": nombre, "union": union, "bytes de mezcla": met["bytes_mezcla"],
                  "tareas con datos": dist["tareas_con_datos"] if hay_mezcla else "sin mezcla",
                  "filas de la tarea mas pesada": f"{dist['fraccion_filas_pesada']:.1%}" if hay_mezcla else "sin mezcla",
                  "bytes de la tarea mas pesada": f"{dist['fraccion_de_la_mas_pesada']:.1%}" if hay_mezcla else "sin mezcla"})
    if nombre.startswith("Mezcla forzada"):
        MEDIDA_FORZADA = dist["fraccion_filas_pesada"]
P.configurar(spark)
print(pd.DataFrame(filas).to_string(index=False))
print(f"fraccion predicha para la mezcla forzada: {PREDICHA:.1%} · medida: {MEDIDA_FORZADA:.1%} · ideal con 8 particiones: {1 / 8:.1%}")
iguales = all(abs(r["consumo_total_m3"] - resultados3["Mezcla forzada (umbral -1)"]["consumo_total_m3"]).max() < 1e-6 for r in resultados3.values())
print("resultado identico en las tres alternativas:", iguales)

                      alternativa             union  bytes de mezcla tareas con datos filas de la tarea mas pesada bytes de la tarea mas pesada
             Catalyst por omision BroadcastHashJoin              716       sin mezcla                   sin mezcla                   sin mezcla
       Mezcla forzada (umbral -1)     SortMergeJoin          6862720                2                        78.2%                        82.5%
Sufijo aleatorio (K=8) con mezcla     SortMergeJoin          8655193                8                        25.0%                        25.7%
fraccion predicha para la mezcla forzada: 78.2% · medida: 78.2% · ideal con 8 particiones: 12.5%
resultado identico en las tres alternativas: True


In [52]:
# Autoverificacion del Nivel 3: la prediccion con el mapa clave -> particion coincide con lo medido
assert abs(PREDICHA - MEDIDA_FORZADA) < 1e-3, (PREDICHA, MEDIDA_FORZADA)
assert iguales
print("Nivel 3: OK · prediccion", f"{PREDICHA:.4f}", "frente a medida", f"{MEDIDA_FORZADA:.4f}")

Nivel 3: OK · prediccion 0.7816 frente a medida 0.7816


**Su decisión · Nivel 3 (decisión 7).** Adoptamos **dejar a Catalyst**: `sectores.parquet` tiene 4 filas, se difunde (BroadcastHashJoin) y la mezcla de la unión desaparece (716 B). La fracción **predicha** para la partición más cargada con mezcla forzada fue de **78,2 %** (S-SUR, S-NORTE y S-OESTE caen en la partición 7 por `hash(sector) mod 8`), y la **medida** también de **78,2 %**, frente al 12,5 % ideal. Bytes de mezcla: forzada 6 862 720 B, con 2 tareas con datos; sufijo aleatorio (K = 8) 8 655 193 B, con la tarea más pesada en el 25,0 %. El sufijo equilibra la carga, pero mueve más bytes. Lo usaríamos solo si la tabla pequeña dejara de caber en la difusión y la tarea más pesada superara unos 128 MiB de mezcla.

## Reto de diseño · 30 minutos · opcional · Emprendimiento Sostenible

**Situación.** La gerencia pregunta si conviene cambiar el diseño físico de la capa. Escriban un **memo de una página** para una persona de finanzas: partición por día, por mes o ninguna para el cierre diario proyectado a tres años, con una tabla de tres diseños (archivos y bytes leídos por cierre, tamaño por archivo y costo de migrar) respaldada con los planes, una recomendación y la cifra que la cambiaría. La competencia es **Emprendimiento Sostenible**: viabilidad económica y uso eficiente de recursos de una decisión técnica sustentada en evidencia propia. Primero un ejercicio de código, luego la tabla.

| Decisión | Opciones | Criterios para decidir | Consecuencia de decidir mal | Qué debe justificarse |
|---|---|---|---|---|
| 1 · Diseño físico de la capa | Por día · por mes · sin partición | Consulta dominante (cierre diario o reporte del mes), tamaño de archivo, archivos y bytes leídos por consulta, costo de reescribir la historia | Leer toda la historia cada día, o pagar la reescritura de la historia sin que el cambio ahorre nada | Archivos y bytes leídos por cierre con cada diseño (medidos o calculados con su supuesto), costo de migrar y punto de equilibrio |

In [53]:
# Reto: cuantos archivos-dia leeria, sin poda, un cierre diario durante D dias si la capa parte con H dias cargados?
def archivos_acumulados(H, D):
    """Suma de los archivos que se leen en los cierres de los dias H+1 a H+D sin poda: el cierre del dia H+k
    abre H+k archivos, asi que el total es D*H + D*(D+1)/2."""
    return D * H + D * (D + 1) // 2


In [54]:
# Reto: la capa proyectada a tres anios con SUS cifras (dos disenos que ya midieron: por dia y por mes)
assert archivos_acumulados(120, 365) == 110_595 and archivos_acumulados(120, 1095) == 731_460
D3 = 1095
kib_mes = medidas_mes["por mes · cierre del dia"]["tamano_archivos_leidos_b"] / 1024
proy = pd.DataFrame([
    {"diseno": "por dia, filtro por la ruta", "archivos por cierre": 1, "KiB por cierre": round(kib_dia, 1), "GiB en 3 anios": round(D3 * kib_dia / 2**20, 3)},
    {"diseno": "por dia, filtro por fecha (sin poda)", "archivos por cierre": "creciente", "KiB por cierre": "creciente", "GiB en 3 anios": round(archivos_acumulados(120, D3) * kib_dia / 2**20, 2)},
    {"diseno": "por mes (cierre del dia)", "archivos por cierre": 1, "KiB por cierre": round(kib_mes, 1), "GiB en 3 anios": round(D3 * kib_mes / 2**20, 3)},
])
print(proy.to_string(index=False))
print("Con kib_mes de un mes completo, el diseno por mes es una COTA SUPERIOR para el cierre: el mes en curso solo tiene los dias ya cargados.")

                              diseno archivos por cierre KiB por cierre  GiB en 3 anios
         por dia, filtro por la ruta                   1           36.7           0.038
por dia, filtro por fecha (sin poda)           creciente      creciente          25.620
            por mes (cierre del dia)                   1          818.2           0.854
Con kib_mes de un mes completo, el diseno por mes es una COTA SUPERIOR para el cierre: el mes en curso solo tiene los dias ya cargados.


## Antes de entregar

Suban al repositorio del equipo: `pipeline/planes_s17.py` completo, `bitacora_planes.jsonl`, `hipotesis_s17.json`, `planes/`, `planes_nivel1.csv`, `evidencia_nivel2_<equipo>.md`, `entregables/t17/` (informe, decisiones, costo y glosario), el contraste del sub-paso A y el índice de evidencias con su README. La celda siguiente comprueba lo que se puede comprobar con código.

In [55]:
# Lista de entrega: lo que se puede comprobar con codigo
esperados = ["pipeline/planes_s17.py", "bitacora_planes.jsonl", "hipotesis_s17.json", f"evidencia_nivel2_{EQUIPO}.md", "planes_nivel1.csv"]
esperados += [f"planes/plan_ampliacion_{c}.txt" for c in P.CONSULTAS]
for e in esperados:
    print(("ok    " if (RAIZ / e).exists() else "FALTA ") + e)
assert all((RAIZ / e).exists() for e in esperados)
assert P.hash_arbol(LAGO_BASE) == HASH_LAGO_ANTES
print("Entreguen ademas: entregables/t17/ (informe, decisiones, costo, glosario) e indice_evidencias.md, y corran la etapa t17 hasta 18 de 18.")

ok    pipeline/planes_s17.py
ok    bitacora_planes.jsonl
ok    hipotesis_s17.json
ok    evidencia_nivel2_equipo-1.md
ok    planes_nivel1.csv
ok    planes/plan_ampliacion_A.txt
ok    planes/plan_ampliacion_A-fecha.txt
ok    planes/plan_ampliacion_B.txt
ok    planes/plan_ampliacion_C.txt
ok    planes/plan_ampliacion_C-forzada.txt
ok    planes/plan_ampliacion_C-aqe.txt


Entreguen ademas: entregables/t17/ (informe, decisiones, costo, glosario) e indice_evidencias.md, y corran la etapa t17 hasta 18 de 18.


In [56]:
# Cierre: spark.stop() detiene la sesion, pero la JVM sigue viva hasta que se cierre el nucleo del cuaderno.
# Antes de ejecutar un script que abra Spark (por ejemplo planes_s17.py), cierren el nucleo: Kernel > Shut Down Kernel.
spark.stop()
print("sesion detenida")

sesion detenida
